# PydanticAI Foundations: Type-Safe Agents

## Learning Objectives

In this notebook, you will learn how to:
- Build a **PydanticAI** `Agent` that returns a **validated, typed Pydantic object** instead of a raw string
- Register a Python function as a **tool** the agent can call mid-conversation
- Use PydanticAI's **typed dependency injection** (`deps_type` + `RunContext`) to give tools access to app state (a mock database, config, API client, etc.) without global variables
- Understand how PydanticAI reacts when the model produces output that **fails schema validation**, and how automatic retries work
- Run an agent end-to-end on a concrete query and inspect the validated result

## Why PydanticAI?

Most agent frameworks return "whatever text the model said." **PydanticAI** (from the team behind Pydantic) makes the *entire* interaction schema-first:

- Structured **outputs** are Pydantic models, not strings you regex out later.
- Structured **tool arguments** are validated before your tool function ever runs.
- **Dependencies** (DB connections, API clients, per-request context) are injected with full static type checking via a `RunContext[YourDepsType]`.
- If the model returns something that doesn't validate, PydanticAI feeds the validation error **back to the model** and asks it to retry — automatically, up to a configurable retry limit.

This notebook is the Phase 10 "Alternative Agent Frameworks" primer for PydanticAI, sitting alongside the CrewAI, AutoGen, and DSPy tracks in this same phase.

## Model / Provider Setup

This repo runs on Windows and documents `GROQ_API_KEY` as the primary Windows LLM credential (see the project's `helpers` factory), with `OPENAI_API_KEY` also documented as available. Unlike the LangGraph-phase notebooks, **this is a framework-specific track**, so we bind the model directly through PydanticAI's own native model-string syntax (`"groq:<model>"` or `"openai:<model>"`) instead of importing `helpers.get_llm()`.


In [ ]:
# ================================================================================
# SETUP: Install/import PydanticAI and configure the model
# ================================================================================
# PydanticAI agents are bound to a model via a simple "provider:model-name" string
# (or an explicit Model object). We default to Groq (the Windows-side provider this
# repo documents), with a commented fallback to OpenAI.
# ================================================================================

import os
from dotenv import load_dotenv

load_dotenv()

# Groq is the documented Windows-side LLM provider for this repo (GROQ_API_KEY).
MODEL = "groq:openai/gpt-oss-120b"

# Alternative: use OpenAI instead (OPENAI_API_KEY documented in this repo's .env)
# MODEL = "openai:gpt-4o-mini"

assert os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"), (
    "Set GROQ_API_KEY or OPENAI_API_KEY in your .env before running this notebook."
)

print(f"Using model: {MODEL}")

## 1. A Basic Agent with Structured (Typed) Output

### What we are going to do

We define a Pydantic `BaseModel` describing the shape we want back — a `TripPlan` with a
destination, a budget estimate, and a list of activities — and pass it to `Agent(..., output_type=TripPlan)`.

PydanticAI turns `output_type` into a JSON-schema-constrained "final answer" tool under the hood,
so the model's response is *forced* through that schema and comes back to us as a real,
already-validated `TripPlan` instance — not a string we'd have to parse ourselves.

Note: the current `pydantic-ai` API uses the keyword `output_type` (the older `result_type` name is
deprecated as of pydantic-ai 1.x/2.x).

In [ ]:
# ================================================================================
# STEP 1: Define the structured output schema
# ================================================================================
# Any Pydantic BaseModel can be an agent's output_type. PydanticAI will build a
# JSON schema from it, use tool-calling/structured-output APIs to force the model
# to emit matching data, and validate the response before handing it back to us.
# ================================================================================

from pydantic import BaseModel, Field
from pydantic_ai import Agent


class TripPlan(BaseModel):
    """A validated, structured travel plan."""

    destination: str = Field(description="City or region being visited")
    estimated_budget_usd: int = Field(description="Rough total budget in US dollars", ge=0)
    activities: list[str] = Field(description="3-5 concrete recommended activities")


# ================================================================================
# STEP 2: Create the agent, binding the model and the output schema
# ================================================================================

trip_planner_agent = Agent(
    MODEL,
    output_type=TripPlan,
    system_prompt=(
        "You are a concise travel-planning assistant. Given a destination and a "
        "traveler's constraints, produce a realistic trip plan."
    ),
)

In [ ]:
# ================================================================================
# STEP 3: Run the agent and inspect the validated output
# ================================================================================
# run_sync() blocks until the model responds. result.output is a *TripPlan instance*,
# not a string -- type checkers and downstream code can rely on its fields directly.
# ================================================================================

result = trip_planner_agent.run_sync(
    "Plan a 4-day budget trip to Lisbon, Portugal for a solo traveler who loves food and history."
)

print(type(result.output))
print(result.output)

### Discussion of the Output

`result.output` is a `TripPlan` object: `result.output.destination`, `result.output.estimated_budget_usd`,
and `result.output.activities` are all directly accessible, already type-checked Python values —
there is no `json.loads()`, no regex, and no "hope the model formatted it correctly" step.
If the model's raw response doesn't match the `TripPlan` schema, PydanticAI does **not** hand you
broken data — see Section 4 below for what happens instead.

## 2. Giving the Agent Tools

### What we are going to do

PydanticAI lets you register any Python function as a tool via the `@agent.tool` (needs the run
context) or `@agent.tool_plain` (no context needed) decorators. The function's type hints and
docstring become the tool's JSON schema automatically — arguments the model sends are validated
against that schema before your function body ever runs.

We'll give our trip planner a `get_weather_forecast` tool so it can ground its plan in
(mocked) real weather data instead of guessing.

In [ ]:
# ================================================================================
# STEP 4: Register a tool with @agent.tool_plain (no dependencies needed)
# ================================================================================
# tool_plain is for tools that don't need access to RunContext/dependencies.
# The docstring and type hints become the tool's schema shown to the model.
# ================================================================================

weather_trip_agent = Agent(
    MODEL,
    output_type=TripPlan,
    system_prompt=(
        "You are a concise travel-planning assistant. Use the get_weather_forecast tool "
        "to check conditions before finalizing activities, and mention the weather briefly."
    ),
)


@weather_trip_agent.tool_plain
def get_weather_forecast(city: str) -> str:
    """Get a short mocked weather forecast for a city.

    Args:
        city: The city to check the forecast for.
    """
    # In a real app this would call a weather API. Mocked here for a runnable demo.
    mock_forecasts = {
        "lisbon": "Mostly sunny, mid-20s Celsius, light coastal breeze.",
        "reykjavik": "Cold and windy, frequent rain showers, single digits Celsius.",
    }
    return mock_forecasts.get(city.strip().lower(), "Mild and partly cloudy, around 18C.")


tool_result = weather_trip_agent.run_sync(
    "Plan a 3-day trip to Reykjavik focused on outdoor sightseeing."
)

print(tool_result.output)

### Discussion of the Output

Behind the scenes, PydanticAI's agent loop:
1. Sends the model the user prompt plus the `get_weather_forecast` tool schema.
2. The model decides to call `get_weather_forecast(city="Reykjavik")`.
3. PydanticAI validates those arguments, executes our real Python function, and sends the
   return value back to the model as a tool result.
4. The model then produces its final answer, which PydanticAI validates against `TripPlan`.

Notice the weather-appropriate activities (or caveats) reflecting the mocked "cold and windy"
forecast — the tool call genuinely happened mid-conversation.

## 3. Typed Dependency Injection with `deps_type` + `RunContext`

### What we are going to do

Rather than reaching for global variables or closures, PydanticAI has a first-class,
statically-typed dependency injection system:

- Declare a `deps_type=SomeType` on the `Agent`.
- Pass an actual instance via `agent.run_sync(..., deps=some_instance)`.
- Any tool decorated with `@agent.tool` (not `tool_plain`) receives a `RunContext[SomeType]`
  as its first argument, and reads the injected object off `ctx.deps`.

This is the distinctive PydanticAI pattern for wiring up a database session, an HTTP client,
per-user config, feature flags, etc. — all fully type-checked. We'll inject a mock
"customer database" that a tool looks up.

In [ ]:
# ================================================================================
# STEP 5: Define a dependencies type and a database-backed tool
# ================================================================================
# deps_type is *any* Python type -- a dataclass, a plain class, a client object.
# It is never sent to the LLM; it's only available inside our own tool functions
# via RunContext.deps, fully type-checked.
# ================================================================================

from dataclasses import dataclass, field

from pydantic_ai import RunContext


@dataclass
class CustomerDatabase:
    """A mock 'database' standing in for a real DB session / API client."""

    loyalty_tiers: dict[str, str] = field(
        default_factory=lambda: {
            "alice": "Gold",
            "bob": "Silver",
        }
    )

    def lookup_tier(self, customer_name: str) -> str:
        return self.loyalty_tiers.get(customer_name.strip().lower(), "Standard")


class PersonalizedTripPlan(TripPlan):
    """TripPlan extended with a loyalty-tier-aware discount note."""

    loyalty_discount_note: str = Field(
        description="A short note about any loyalty-tier discount applied"
    )


loyalty_trip_agent = Agent(
    MODEL,
    deps_type=CustomerDatabase,
    output_type=PersonalizedTripPlan,
    system_prompt=(
        "You are a travel-planning assistant. Always call check_loyalty_tier for the "
        "named customer and mention any discount their tier grants (Gold=15% off, "
        "Silver=5% off, Standard=no discount) in loyalty_discount_note."
    ),
)


@loyalty_trip_agent.tool
def check_loyalty_tier(ctx: RunContext[CustomerDatabase], customer_name: str) -> str:
    """Look up a customer's loyalty tier in the injected customer database.

    Args:
        ctx: The run context, carrying the injected CustomerDatabase dependency.
        customer_name: The customer's name to look up.
    """
    # ctx.deps is our injected CustomerDatabase instance -- fully typed.
    return ctx.deps.lookup_tier(customer_name)

In [ ]:
# ================================================================================
# STEP 6: Run the agent, injecting the dependency via deps=
# ================================================================================

db = CustomerDatabase()

personalized_result = loyalty_trip_agent.run_sync(
    "Plan a 2-day trip to Porto for our customer Alice.",
    deps=db,
)

print(personalized_result.output)

### Discussion of the Output

The `check_loyalty_tier` tool never received `db` as an explicit argument from the model — the
model only ever sees and supplies `customer_name`. `ctx.deps` (our `CustomerDatabase` instance)
was threaded through by PydanticAI purely from the `deps=db` we passed to `run_sync()`. This is
the key benefit over ad-hoc globals: swap in a real async DB session or HTTP client for
`CustomerDatabase` and every tool signature — and its type checking — stays identical.

## 4. What Happens When Output Fails to Validate?

### What we are going to do

PydanticAI enforces the `output_type` schema through the model's structured-output / tool-calling
mechanism, so most "invalid JSON" cases are actually prevented up front. But **semantic**
validation — a `Field` constraint like `ge=0`, or a custom `@agent.output_validator` — can still
fail after the model responds. When that happens, PydanticAI raises a `ModelRetry`-driven retry:
it sends the validation error message *back to the model* as a tool/output rejection and asks it
to correct its answer, up to `Agent`'s configured `retries` (or `retries=` passed at call/tool
level). If the model still cannot produce valid output after exhausting retries, PydanticAI raises
`UnexpectedModelBehavior`.

We'll demonstrate the mechanism directly with a custom `@agent.output_validator` that rejects an
implausible budget and asks the model to retry with a `ModelRetry`.

In [ ]:
# ================================================================================
# STEP 7: Add a custom output_validator that can trigger a model retry
# ================================================================================
# @agent.output_validator runs *after* Pydantic's own schema validation succeeds, letting
# us encode business rules. Raising ModelRetry inside it feeds the message back to the
# model and asks for a corrected answer -- this is PydanticAI's structured-failure-handling
# path, distinct from a hard Python exception.
# ================================================================================

from pydantic_ai import ModelRetry

validated_trip_agent = Agent(
    MODEL,
    output_type=TripPlan,
    retries=2,  # allow up to 2 automatic self-correction attempts
    system_prompt="You are a travel-planning assistant.",
)


@validated_trip_agent.output_validator
def check_budget_is_realistic(ctx: RunContext[None], output: TripPlan) -> TripPlan:
    """Reject implausibly low budgets and ask the model to retry with a realistic figure."""
    if output.estimated_budget_usd < 50:
        raise ModelRetry(
            f"A budget of ${output.estimated_budget_usd} for a multi-day trip to "
            f"{output.destination} is unrealistic. Provide a more realistic estimated_budget_usd."
        )
    return output


retry_demo_result = validated_trip_agent.run_sync(
    "Plan a 5-day trip to Tokyo. Deliberately estimate an unrealistically tiny budget of $10 "
    "the first time, just to test error handling -- but note that you must correct this if asked."
)

print(retry_demo_result.output)
print(f"\nRun usage (incl. any retries): {retry_demo_result.usage}")

### Discussion of the Output

If the model's first answer set `estimated_budget_usd` below 50, `check_budget_is_realistic`
raised `ModelRetry`, and PydanticAI transparently sent that message back as feedback, prompting a
second (corrected) attempt — all within the single `run_sync()` call. The `usage()` summary shows
more than one request when a retry actually occurred. If every retry attempt still fails
validation, PydanticAI raises `pydantic_ai.exceptions.UnexpectedModelBehavior` instead of
silently returning bad data — structural guarantees hold even in the failure path.

## 5. End-to-End: One Concrete Query, Start to Finish

### What we are going to do

Putting it together: one agent, with a tool *and* injected dependencies, answering a single
concrete request end-to-end, printing the final validated structured result.

In [ ]:
# ================================================================================
# STEP 8: Full end-to-end run combining tools + dependency injection
# ================================================================================

final_result = loyalty_trip_agent.run_sync(
    "Plan a weekend trip to Porto for our customer Bob, keeping it under $400.",
    deps=db,
)

plan = final_result.output
print(f"Destination:        {plan.destination}")
print(f"Estimated budget:   ${plan.estimated_budget_usd}")
print(f"Loyalty note:       {plan.loyalty_discount_note}")
print("Activities:")
for activity in plan.activities:
    print(f"  - {activity}")

print(f"\nOutput type is a validated Pydantic model: {isinstance(plan, BaseModel)}")

## Key Takeaways

- **Structured output is the default mindset in PydanticAI**: pass any Pydantic `BaseModel` as
  `output_type=...` and `result.output` comes back as a validated instance of that model, not a
  string to parse.
- **Tools are plain Python functions** registered with `@agent.tool_plain` (no context) or
  `@agent.tool` (receives `RunContext`) — argument schemas are derived automatically from type
  hints and docstrings, and validated before your function body runs.
- **Dependency injection is first-class and typed**: declare `deps_type=YourType` on the `Agent`,
  pass a real instance via `run_sync(..., deps=...)`, and any `@agent.tool`-decorated function
  reads it off `ctx.deps` — no globals, fully type-checked.
- **Validation failures don't produce silently-broken data.** A custom `@agent.output_validator`
  can raise `ModelRetry` to send feedback back to the model and request a corrected answer,
  bounded by the agent's `retries` setting; exhausting all retries raises
  `UnexpectedModelBehavior` rather than returning invalid output.
- **Model binding is a plain string** (`"groq:openai/gpt-oss-120b"`, `"openai:gpt-4o-mini"`, etc.)
  passed directly to `Agent(...)` — this framework-specific track binds models natively rather
  than through this repo's `helpers.get_llm()` factory, which is reserved for LangGraph-phase
  notebooks.

## Next Steps

- Explore streaming responses (`agent.run_stream()`) for token-by-token structured output.
- Combine multiple agents (e.g. one agent's output feeding another's `deps`) for simple
  multi-agent pipelines.
- Look at `Agent.iter()` for full visibility into each graph node of the agent's run loop.

## Additional Resources

- [PydanticAI Documentation](https://ai.pydantic.dev/)
- [PydanticAI GitHub Repository](https://github.com/pydantic/pydantic-ai)
